[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/adv-10b-post-training-data-dedup-decontamination-masking-packing.ipynb)

# Advanced Discussion 10 (continued) — the data behind post-training

MinHash and LSH near-duplicate removal, n-gram versus embedding decontamination, loss masking, and sequence packing.

**Optional advanced-discussion lab** for Generative AI and Large Language Models with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier; CPU is enough unless a note says otherwise.

Downloads a small embedding model (about 90 MB); runs in about a minute.

In [ ]:
%pip install -q numpy sentence-transformers tiktoken

In [ ]:
import numpy as np, random, re, hashlib, itertools
random.seed(0); np.random.seed(0)

## 1. a small corpus with planted near-duplicates: 600 distinct 80-word texts, then 200 copies with 1-8 random word edits

In [ ]:
VOCAB = ["w%d" % i for i in range(3000)]; zipf = 1 / np.arange(1, len(VOCAB) + 1); zipf /= zipf.sum()
def perturb(s, k):
    toks = s.split()
    for _ in range(k):
        op = random.choice(["ins", "del", "swap"]); i = random.randrange(len(toks))
        if op == "ins": toks.insert(i, random.choice(VOCAB))
        elif op == "del": toks.pop(i)
        else: j = random.randrange(len(toks)); toks[i], toks[j] = toks[j], toks[i]
    return " ".join(toks)
rs = np.random.RandomState(0)
docs = [" ".join(rs.choice(VOCAB, 80, p=zipf)) for _ in range(600)]
truth = set()
for _ in range(200):
    src = random.randrange(600); d = perturb(docs[src], random.randint(1, 8)); docs.append(d); truth.add((src, len(docs) - 1))
print("documents:", len(docs), "; planted near-duplicate pairs:", len(truth))

def shingles(s, k=3):
    t = s.split(); return {" ".join(t[i:i + k]) for i in range(max(1, len(t) - k + 1))}
def jaccard(a, b): return len(a & b) / len(a | b)
SH = [shingles(d) for d in docs]

## 2. exact all-pairs Jaccard (the slow reference) and what threshold to use

In [ ]:
n = len(docs); pairs_exact = []
for i in range(n):
    for j in range(i + 1, n):
        jac = jaccard(SH[i], SH[j])
        if jac >= 0.3: pairs_exact.append((i, j, jac))
print("all-pairs comparisons: %d" % (n * (n - 1) // 2))
truth_pairs = {(min(a, b), max(a, b)) for a, b in truth}
print("Jaccard of the planted pairs: min %.2f, median %.2f" % (min(jaccard(SH[a], SH[b]) for a, b in truth_pairs), np.median([jaccard(SH[a], SH[b]) for a, b in truth_pairs])))
for th in (0.3, 0.5, 0.7):
    found = {(i, j) for i, j, s in pairs_exact if s >= th}
    print("  exact Jaccard >= %.1f: flagged %d pairs, recall of planted %.2f, precision %.2f" % (th, len(found), len(found & truth_pairs) / len(truth_pairs), len(found & truth_pairs) / max(1, len(found))))

## 3. MinHash + LSH banding: same job without comparing every pair

In [ ]:
def h32(s): return int(hashlib.md5(s.encode()).hexdigest()[:8], 16)
def signature(sh, k):
    """MinHash: for each of k salted hash functions keep the smallest hash over the document's shingles."""
    return [min(h32(str(salt) + "|" + x) for x in sh) for salt in range(k)]
K = 128; sig = [signature(s, K) for s in SH]
est_err = [abs(np.mean(np.array(sig[a]) == np.array(sig[b])) - jaccard(SH[a], SH[b])) for a, b in list(truth_pairs)[:100]]
print("\nMinHash with %d hashes: mean |estimated - true Jaccard| on 100 planted pairs = %.3f" % (K, np.mean(est_err)))
def lsh_candidates(sig, bands, rows):
    buckets = {}
    for idx, s in enumerate(sig):
        for b in range(bands):
            key = (b, tuple(s[b * rows:(b + 1) * rows])); buckets.setdefault(key, []).append(idx)
    cands = set()
    for v in buckets.values():
        for a, b in itertools.combinations(v, 2): cands.add((min(a, b), max(a, b)))
    return cands
print("bands x rows   candidates   recall of planted   precision (after checking Jaccard >= 0.5)   fraction of all pairs examined")
for bands, rows in ((16, 8), (32, 4), (64, 2)):
    c = lsh_candidates(sig, bands, rows); kept = {(i, j) for i, j in c if jaccard(SH[i], SH[j]) >= 0.5}
    print("  %2d x %d        %6d         %.2f                %.2f                                       %.4f" % (bands, rows, len(c), len(c & truth_pairs) / len(truth_pairs), len(kept & truth_pairs) / max(1, len(kept)), len(c) / (n * (n - 1) / 2)))
print("probability a pair with Jaccard s becomes a candidate = 1 - (1 - s^r)^b :")
for bands, rows in ((16, 8), (32, 4), (64, 2)):
    print("  %2d x %d: " % (bands, rows) + "  ".join("s=%.1f -> %.3f" % (s, 1 - (1 - s ** rows) ** bands) for s in (0.2, 0.4, 0.6, 0.8)))

## 4. decontamination: n-gram overlap catches copies, misses paraphrases; embeddings catch more

In [ ]:
TEST = ["what is the capital of the country whose flag has a red circle on white", "a train leaves at noon travelling at sixty miles per hour how far has it gone by three",
        "explain why the sky appears blue during the day and red at sunset", "write a function that returns the second largest number in a list without sorting it"]
train = ["general knowledge quiz " + t for t in TEST[:1]]                                               # verbatim copy inside a longer text
train += ["a train departs at 12:00 going 60 mph; how many miles has it covered by 3 pm", "why is the sky blue in daytime yet red when the sun sets, explain"]  # paraphrases
train += ["what is the boiling point of water at sea level", "sort a list of strings by length in python", "describe how photosynthesis works in plants"]
def ngrams(s, n=8):
    t = re.findall(r"\w+", s.lower()); return {tuple(t[i:i + n]) for i in range(len(t) - n + 1)}
print("\n8-gram overlap with each test item (does any training text contain one of its 8-grams?)")
from sentence_transformers import SentenceTransformer
enc = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2"); ET = enc.encode(TEST, normalize_embeddings=True); ER = enc.encode(train, normalize_embeddings=True)
sim = ET @ ER.T
for i, t in enumerate(TEST):
    ov = any(ngrams(t) & ngrams(r) for r in train); print("  test %d: 8-gram match %-5s | best cosine %.2f with training text %d" % (i, ov, sim[i].max(), sim[i].argmax()))
print("cosine of unrelated pairs (median over all test x train pairs not matched above): %.2f" % np.median(sim))

## 5. chat-template loss masking: which tokens should the loss see?

In [ ]:
import tiktoken
tok = tiktoken.get_encoding("cl100k_base")
convs = [("Summarise the refund policy in one sentence.", "Refunds are available within 30 days of delivery for damaged or wrong items."),
         ("Translate 'good morning' into French.", "Bonjour."),
         ("List three causes of a delayed shipment and how to check each one, briefly.", "1. Customs hold - check the tracking page. 2. Carrier backlog - contact the carrier. 3. Address error - verify the address on the order."),
         ("What is 17 times 23?", "391.")]
tot_p = tot_a = 0
print("\nper example: prompt tokens, answer tokens, share of tokens the loss sees when the prompt is masked")
for p, a in convs:
    np_, na = len(tok.encode(p)), len(tok.encode(a)); tot_p += np_; tot_a += na; print("  prompt %3d  answer %3d  loss sees %.0f%%" % (np_, na, 100 * na / (np_ + na)))
print("  overall: %.0f%% of tokens carry loss when only the answer is trained on" % (100 * tot_a / (tot_p + tot_a)))

## 6. packing: padding waste vs packed sequences

In [ ]:
lens = np.clip(np.random.RandomState(0).lognormal(mean=4.6, sigma=0.7, size=2000).astype(int), 8, 1024)
MAXLEN = 1024
padded_tokens = len(lens) * MAXLEN; useful = lens.sum()
bins = []
for L in sorted(lens, reverse=True):                                                             # first-fit decreasing
    for b in range(len(bins)):
        if bins[b] + L <= MAXLEN: bins[b] += L; break
    else: bins.append(L)
print("\n2000 examples, median length %d, max length %d, sequence length %d" % (np.median(lens), lens.max(), MAXLEN))
print("  one example per sequence (padded): %d sequences, %.0f%% of positions are padding" % (len(lens), 100 * (1 - useful / padded_tokens)))
print("  packed first-fit: %d sequences, %.1f%% of positions are padding; %.1fx fewer sequences" % (len(bins), 100 * (1 - useful / (len(bins) * MAXLEN)), len(lens) / len(bins)))

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.